In [1]:
# ─── Input / output paths ─────────────────────────────────────────────────────
INPUT_FILE  = r"C:\Users\IvanVelilla\Mobility Lab Limited\Projects - 1034 - Western LX Aimsun\New North Road\Data\Full Network Summary and Open-Close Data_2025 Analysis.xlsm"
OUTPUT_FILE = r"C:\Users\IvanVelilla\Mobility Lab Limited\Projects - 1034 - Western LX Aimsun\New North Road\Data\cp_boomtime_pm.txt"

# ─── Control plan name (first line of the output file) ────────────────────────
CP_NAME = "boomtime_pm"

# ─── Sites to include ─────────────────────────────────────────────────────────
# Run the Setup cell below to see all available site codes.
SITES = ["GE", "MO", "AS", "RO", "WO", "SJ", "CH", "SG"]

# ─── Time window ──────────────────────────────────────────────────────────────
START_TIME = "15:00"   # HH:MM
END_TIME   = "19:00"   # HH:MM

# ─── Date selection ───────────────────────────────────────────────────────────
# Provide one or more dates to use a specific day's closures.
# Set to None to average closure patterns across all weekdays in the workbook.
DATES = ["2025/03/11"]
# DATES = ["2025/03/10"]                          # single day
# DATES = ["2025/03/10", "2025/03/11", "2025/03/12"]  # several days averaged

# ─── Averaging gap threshold ──────────────────────────────────────────────────
# When averaging multiple days, closures within this many seconds of each other
# are grouped as the same train slot and their timings averaged.
# Only used when DATES is None or has more than one entry.
GAP_THRESHOLD_S = 900  # 15 minutes

In [2]:
import sys
import openpyxl
from pathlib import Path

sys.path.insert(0, r"C:\Users\IvanVelilla\Documents\Projects\Western LX\src")

from create_cp_boomtime import (
    SITE_CONFIG,
    get_weekday_dates,
    read_closures,
    write_cp,
    _parse_time_arg,
    _fmt_sec,
)

wb = openpyxl.load_workbook(INPUT_FILE, read_only=True, keep_vba=False, data_only=True)
available = [s for s in wb.sheetnames if s in SITE_CONFIG]
wb.close()

print(f"Available sites in: {Path(INPUT_FILE).name}\n")
print(f"  {'Code':<6}  {'Name':<26}  {'Node ID':>12}  {'sig_on':>12}  {'sig_off':>12}  Status")
print(f"  {'-'*6}  {'-'*26}  {'-'*12}  {'-'*12}  {'-'*12}  ------")
for s in available:
    cfg = SITE_CONFIG[s]
    status = "ok" if cfg["node"] != 0 else "WARNING: node ID not set"
    print(f"  {s:<6}  {cfg['name']:<26}  {cfg['node']:>12}  {cfg['sig_on']:>12}  {cfg['sig_off']:>12}  {status}")

Available sites in: Full Network Summary and Open-Close Data_2025 Analysis.xlsm

  Code    Name                             Node ID        sig_on       sig_off  Status
  ------  --------------------------  ------------  ------------  ------------  ------
  GE      George Street                   10260974      10260986      10260987  ok
  MO      Morningside Drive               10267816      10267822      10267823  ok
  AS      Asquith Ave                     89356028      89356031      89356032  ok
  RO      Rossgrove                       89356004      89356009      89356010  ok
  WO      Woodward Road                   10264574      10264578      10264579  ok
  SJ      St Jude Street                  10266940      10266945      10266946  ok
  CH      Chalmers St                     89375286      89375292      89375293  ok
  SG      Saint Georges St                89362210      89371677      89375283  ok
  FR      FR                                     0             0             0  W

In [3]:
window_start = _parse_time_arg(START_TIME)
window_end   = _parse_time_arg(END_TIME)
cycle        = window_end - window_start
date_filter  = set(DATES) if DATES else None
use_average  = date_filter is None or len(date_filter) > 1

# Validate
assert window_end > window_start, "END_TIME must be later than START_TIME"
bad = [s for s in SITES if s not in SITE_CONFIG]
assert not bad, f"Unknown site code(s): {bad} — check SITE_CONFIG in create_cp_boomtime.py"

print(f"Window : {START_TIME} – {END_TIME}  ({cycle}s = {cycle//3600}h {(cycle%3600)//60}m)")
print(f"Mode   : {'average weekday' if use_average else 'specific date(s): ' + str(sorted(date_filter))}")
print(f"Sites  : {SITES}\n")

# Load closures from workbook
wb = openpyxl.load_workbook(INPUT_FILE, read_only=True, keep_vba=False, data_only=True)
closures_per_site = {}
for site in SITES:
    ws = wb[site]
    if date_filter is None:
        dates = set(get_weekday_dates(ws))
        print(f"  {site}: {len(dates)} weekday(s) found")
    else:
        dates = date_filter
        print(f"  {site}: using date(s) {sorted(dates)}")
    closures_per_site[site] = read_closures(ws, dates)
wb.close()

# Write control plan
out_path = Path(OUTPUT_FILE)
out_path.parent.mkdir(parents=True, exist_ok=True)

print(f"\nGenerating control plan...")
write_cp(
    out_path=out_path,
    cp_name=CP_NAME,
    sites=SITES,
    window_start=window_start,
    window_end=window_end,
    closures_per_site=closures_per_site,
    use_average=use_average,
    gap_threshold=GAP_THRESHOLD_S,
)

print(f"\nWritten to: {out_path}")

Window : 15:00 – 19:00  (14400s = 4h 0m)
Mode   : specific date(s): ['2025/03/11']
Sites  : ['GE', 'MO', 'AS', 'RO', 'WO', 'SJ', 'CH', 'SG']

  GE: using date(s) ['2025/03/11']
  MO: using date(s) ['2025/03/11']
  AS: using date(s) ['2025/03/11']
  RO: using date(s) ['2025/03/11']
  WO: using date(s) ['2025/03/11']
  SJ: using date(s) ['2025/03/11']
  CH: using date(s) ['2025/03/11']
  SG: using date(s) ['2025/03/11']

Generating control plan...

Written to: C:\Users\IvanVelilla\Mobility Lab Limited\Projects - 1034 - Western LX Aimsun\New North Road\Data\cp_boomtime_pm.txt


  GE: 47 closures, total=04:00:00 vs cycle=04:00:00
  MO: 29 closures, total=04:00:00 vs cycle=04:00:00
  AS: 45 closures, total=04:00:00 vs cycle=04:00:00
  RO: 45 closures, total=04:00:00 vs cycle=04:00:00
  WO: 46 closures, total=04:00:00 vs cycle=04:00:00
  SJ: 33 closures, total=04:00:00 vs cycle=04:00:00
  CH: 31 closures, total=04:00:00 vs cycle=04:00:00
  SG: 32 closures, total=04:00:00 vs cycle=04:00:00
